In [1]:
# Cell 0: Add project root to path so we can import from parent directory
import sys
import os
from pathlib import Path

# Try multiple strategies to find the project root
def find_project_root():
    # Strategy 1: Look for private_path_query_utils.py going up from cwd
    path = Path.cwd()
    for _ in range(5):  # Check up to 5 levels
        if (path / "private_path_query_utils.py").exists():
            return path
        path = path.parent
    
    # Strategy 2: Use parent of notebooks directory
    cwd = Path.cwd()
    if cwd.name == "notebooks":
        return cwd.parent
    
    # Strategy 3: Hardcoded fallback
    return Path("/Users/joshuamayhugh/Projects/aima-python")

project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Also change working directory to project root for consistency
os.chdir(project_root)
print(f"Project root: {project_root}")
print(f"Working directory: {Path.cwd()}")

Project root: /Users/joshuamayhugh/Projects/aima-python
Working directory: /Users/joshuamayhugh/Projects/aima-python


In [ ]:
# Import all necessary modules
from private_path_query_utils import Vertex, Edge, EdgeState, Graph, GraphPath
from private_path_query_logic import (
    build_bob_q, 
    build_alice_q, 
    build_physics_q,
    build_q,
    ordered_symbols, 
    directed_pairs_from_edges,
    num_bits,
    PosBit,
    Active,
    Allowed,
)
import numpy as np

print("Imports successful!")


Imports successful!


In [ ]:
# Graph-plan style route: 0 -> 1 -> 3
plan_edges = [
    Edge(vertices[0], vertices[1], EdgeState.TRAVERSABLE),
    Edge(vertices[1], vertices[3], EdgeState.TRAVERSABLE),
]
plan = GraphPath(start=vertices[0], moves=plan_edges)

print("\nGraphPath (edge sequence):")
for step, e in enumerate(plan.moves):
    print(f"  step {step}: {e.vertex1.id} -> {e.vertex2.id}")

T = len(plan.moves)
V = len(vertices)

# Optional compact domain from edges present in graph.
compact_pairs = directed_pairs_from_edges(edges, V)
syms = ordered_symbols(T=T, V=V, directed_pairs=compact_pairs)

# This now prints CNF clauses generated from Bob physics.
q_bob, w_bob = build_bob_q(
    edges=edges,
    T=T,
    V=V,
    symbols=syms,
    directed_pairs=compact_pairs,
    print_cnf_clauses=True,
)

print("\nQ_bob shape:", q_bob.shape)
print("w_bob shape:", w_bob.shape)
print("First 5 rows of Q_bob:")
print(q_bob[:5])
print("First 5 bob clause weights:", w_bob[:5])

Graph with V=4 vertices
Number of bits needed: b = ceil(log2(4)) = 2

Directed edges:
  0 -> 1: TRAVERSABLE
  1 -> 0: TRAVERSABLE
  1 -> 2: BLOCKED
  1 -> 3: TRAVERSABLE
  2 -> 1: BLOCKED
  2 -> 3: TRAVERSABLE
  3 -> 1: TRAVERSABLE
  3 -> 2: TRAVERSABLE


Path-finding problem:
  Start vertex: 0
  Goal vertex:  3
  Time horizon: T = 2
  Valid path:   0 -> 1 -> 3 (uses traversable edges only)
  Invalid path: 0 -> 1 -> 2 -> 3 (edge 1->2 is BLOCKED)

Edge domain (directed pairs): [(0, 1), (1, 2), (1, 3), (2, 3)]

=== Symbol ordering (14 symbols) ===
Position bits: PosBit(t, k) for t in 0..2, k in 0..1
  Time 0: ['PosBit(0, 0)', 'PosBit(0, 1)']
  Time 1: ['PosBit(1, 0)', 'PosBit(1, 1)']
  Time 2: ['PosBit(2, 0)', 'PosBit(2, 1)']

Edge variables:
  Active:  [Active(0, 1), Active(1, 2), Active(1, 3), Active(2, 3)]
  Allowed: [Allowed(0, 1), Allowed(1, 2), Allowed(1, 3), Allowed(2, 3)]


=== Alice's Constraints ===
Alice wants: pos(0) = 0 and pos(T) = pos(2) = 3

Bit encoding (b=2 bits):
  start=0 in binary: 00 -> ~PosBit(0,0) ~PosBit(0,1) 
  goal=3 in binary:  11 -> PosBit(2,0) PosBit(2,1) 
=== build_alice_q: Alice physics CNF clauses ===
alice_physics[0] = ~PosBit(0, 0)
  cnf[0] (from local 0, weight=1.0) = ~PosBit(0, 0)
alice_physics[1] = ~PosBit(0, 1)
  cnf[1] (from local 0, weight=1.0) = ~PosBit(0, 1)
alice_physics[2] = PosBit(2, 0)
  cnf[2] (from local 0, weight=1.0) = PosBit(2, 0)
alice_physics[3] = PosBit(2, 1)
  cnf[3] (from local 0, weight=1.0) = PosBit(2, 1)
Total CNF clauses: 4

Q_alice shape: (4, 28)
  - 4 clauses (rows)
  - 28 columns = 2 * 14 symbols (positive + negative literals)


=== Bob's Edge Constraints ===
Bob knows the state of each edge:
  Edge 0->1 TRAVERSABLE: Active(0,1)=T, Allowed(0,1)=T
  Edge 1->2 BLOCKED:  Active(1,2)=T, Allowed(1,2)=F [SOFT]
  Edge 1->3 TRAVERSABLE: Active(1,3)=T, Allowed(1,3)=T
  Edge 2->3 TRAVERSABLE: Active(2,3)=T, Allowed(2,3)=T

=== build_bob_q: Bob physics CNF clauses ===
bob_physics[0] = Active(0, 1)
  cnf[0] (from local 0, weight=1.0) = Active(0, 1)
bob_physics[1] = Allowed(0, 1)
  cnf[1] (from local 0, weight=1.0) = Allowed(0, 1)
bob_physics[2] = (Allowed(0, 1) ==> Active(0, 1))
  cnf[2] (from local 0, weight=1.0) = (Active(0, 1) | ~Allowed(0, 1))
bob_physics[3] = Active(1, 2)
  cnf[3] (from local 0, weight=1.0) = Active(1, 2)
bob_physics[4] = ~Allowed(1, 2)
  cnf[4] (from local 0, weight=0.3) = ~Allowed(1, 2)
bob_physics[5] = (Allowed(1, 2) ==> Active(1, 2))
  cnf[5] (from local 0, weight=1.0) = (Active(1, 2) | ~Allowed(1, 2))
bob_physics[6] = Active(1, 3)
  cnf[6] (from local 0, weight=1.0) = Active(1, 3)
bob_physics[7]

=== Physics Constraints ===
With V=4 and b=2 bits, valid codes are 0..3
V is a power of 2, no invalid codes to forbid

=== build_physics_q: Physics CNF clauses ===
physics[0] = ((~PosBit(0, 0) & ~PosBit(0, 1)) ==> ((PosBit(1, 0) & ~PosBit(1, 1)) | (~PosBit(1, 0) & ~PosBit(1, 1))))
  cnf[0] (from local 0, weight=1.0) = (~PosBit(1, 0) | PosBit(1, 0) | PosBit(0, 0) | PosBit(0, 1))
  cnf[1] (from local 1, weight=1.0) = (~PosBit(1, 1) | PosBit(1, 0) | PosBit(0, 0) | PosBit(0, 1))
  cnf[2] (from local 2, weight=1.0) = (~PosBit(1, 0) | ~PosBit(1, 1) | PosBit(0, 0) | PosBit(0, 1))
  cnf[3] (from local 3, weight=1.0) = (~PosBit(1, 1) | ~PosBit(1, 1) | PosBit(0, 0) | PosBit(0, 1))
physics[1] = ((PosBit(0, 0) & ~PosBit(0, 1)) ==> (((~PosBit(1, 0) & PosBit(1, 1)) | (PosBit(1, 0) & PosBit(1, 1))) | (PosBit(1, 0) & ~PosBit(1, 1))))
  cnf[4] (from local 0, weight=1.0) = (PosBit(1, 0) | PosBit(1, 0) | ~PosBit(1, 0) | ~PosBit(0, 0) | PosBit(0, 1))
  cnf[5] (from local 1, weight=1.0) = (~PosBit(1, 1) | 

=== Full Q Matrix ===
Q_physics: 48 rows
Q_alice:   4 rows
Q_bob:     12 rows

Full Q shape: (64, 28)
  - 64 total clauses
  - 28 columns = 2 * 14 (pos + neg literals)

Weight vector shape: (64,)
  Hard clauses (weight=1.0): 63
  Soft clauses (weight=0.1): 1


=== Q Matrix Structure ===
Number of symbols: n = 14
Q has 28 columns = 2 * 14
  Columns 0..13: positive literals
  Columns 14..27: negative literals

Symbol index mapping:
  0: PosBit(0, 0) (positive col=0, negative col=14)
  1: PosBit(0, 1) (positive col=1, negative col=15)
  2: PosBit(1, 0) (positive col=2, negative col=16)
  3: PosBit(1, 1) (positive col=3, negative col=17)
  4: PosBit(2, 0) (positive col=4, negative col=18)
  5: PosBit(2, 1) (positive col=5, negative col=19)
  6: Active(0, 1) (positive col=6, negative col=20)
  7: Active(1, 2) (positive col=7, negative col=21)
  8: Active(1, 3) (positive col=8, negative col=22)
  9: Active(2, 3) (positive col=9, negative col=23)
  10: Allowed(0, 1) (positive col=10, negative col=24)
  11: Allowed(1, 2) (positive col=11, negative col=25)
  12: Allowed(1, 3) (positive col=12, negative col=26)
  13: Allowed(2, 3) (positive col=13, negative col=27)

Example: First Alice clause (start constraint)
  Positive literals: []
  Negative lite

=== Summary: Bit Encoding vs One-Hot ===

Problem size: V=4 vertices, T=2 timesteps, 4 edges

Old one-hot encoding (dense):
  At(t,v):     12
  Move(t,u,v): 32
  Wait(t,v):   8
  Total:       52

New bit encoding (sparse edge domain):
  PosBit(t,k): 6
  Active/Allowed: 8
  Total:       14

Variable reduction: 52 -> 14 (73.1% fewer)

Key insight: AMO (at-most-one) is FREE with bit encoding!
  - Old: O(V²) pairwise clauses per timestep
  - New: 0 clauses (binary arithmetic guarantees uniqueness)


=== Creating Weighted Edges ===

Raw edges (before normalization):
  0 -> 1: OK       weight=1.0
  1 -> 2: OK       weight=1.0
  2 -> 4: OK       weight=1.0
  0 -> 4: BLOCKED  weight=5.0
  1 -> 3: BLOCKED  weight=2.0
  3 -> 4: BLOCKED  weight=0.5

Normalized edges (weights sum to 1.0):
  0 -> 1: OK       weight=0.0952
  1 -> 2: OK       weight=0.0952
  2 -> 4: OK       weight=0.0952
  0 -> 4: BLOCKED  weight=0.4762
  1 -> 3: BLOCKED  weight=0.1905
  3 -> 4: BLOCKED  weight=0.0476

  Total: 1.0000


=== Building Bob's Q with Weighted Edges ===

=== build_bob_q: Bob physics CNF clauses ===
bob_physics[0] = Active(0, 1)
  cnf[0] (from local 0, weight=1.0) = Active(0, 1)
bob_physics[1] = Allowed(0, 1)
  cnf[1] (from local 0, weight=1.0) = Allowed(0, 1)
bob_physics[2] = (Allowed(0, 1) ==> Active(0, 1))
  cnf[2] (from local 0, weight=1.0) = (Active(0, 1) | ~Allowed(0, 1))
bob_physics[3] = Active(0, 4)
  cnf[3] (from local 0, weight=1.0) = Active(0, 4)
bob_physics[4] = ~Allowed(0, 4)
  cnf[4] (from local 0, weight=0.47619047619047616) = ~Allowed(0, 4)
bob_physics[5] = (Allowed(0, 4) ==> Active(0, 4))
  cnf[5] (from local 0, weight=1.0) = (Active(0, 4) | ~Allowed(0, 4))
bob_physics[6] = Active(1, 2)
  cnf[6] (from local 0, weight=1.0) = Active(1, 2)
bob_physics[7] = Allowed(1, 2)
  cnf[7] (from local 0, weight=1.0) = Allowed(1, 2)
bob_physics[8] = (Allowed(1, 2) ==> Active(1, 2))
  cnf[8] (from local 0, weight=1.0) = (Active(1, 2) | ~Allowed(1, 2))
bob_physics[9] = Active(1, 3)
  cnf[9] 